# 01 — 1ジョブ学習

`FOLD` と `SIZE` を指定して1ジョブ実行する。結果はDriveに保存される。
30ジョブ（5 fold × 6 size）を順次実行する。

**GPUランタイム推奨。**

In [ ]:
# ===== ここを変えて実行する =====
FOLD = 1          # 1〜5
SIZE = 15         # 15 / 30 / 60 / 90 / 120 / 149
# ================================

DRIVE_DATA_DIR   = "/content/drive/MyDrive/anglist_data"
DRIVE_LC_DIR     = "/content/drive/MyDrive/anglist_learning_curve"  # folds.json と results/ の場所
EPOCHS     = 100
BATCH_SIZE = 4
RESIZE     = [512, 512]
SIGMA      = 15.0
LR         = 1e-3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/masaki39/anglist /content/anglist
!pip install -q tqdm onnxruntime

In [ ]:
import json, os, random, shutil

# フォールド割り当てを読み込む
with open(os.path.join(DRIVE_LC_DIR, "folds.json")) as f:
    fold_data = json.load(f)

test_ids  = fold_data["folds"][str(FOLD)]
train_ids = [cid for k, ids in fold_data["folds"].items() for cid in ids if k != str(FOLD)]

# 訓練サイズにサブサンプル（再現性のため固定シード）
random.seed(FOLD * 1000 + SIZE)
random.shuffle(train_ids)
train_ids = train_ids[:SIZE]

print(f"Fold {FOLD} | Size {SIZE} | train={len(train_ids)} | test={len(test_ids)}")

# ローカルにコピー
LOCAL_TRAIN = "/content/train_data"
LOCAL_TEST  = "/content/test_data"
for d in [LOCAL_TRAIN, LOCAL_TEST]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

def copy_case(case_id, src_dir, dst_dir):
    for ext in ["_image.npy", "_landmarks.json"]:
        shutil.copy(os.path.join(src_dir, case_id + ext), dst_dir)

for cid in train_ids:
    copy_case(cid, DRIVE_DATA_DIR, LOCAL_TRAIN)
for cid in test_ids:
    copy_case(cid, DRIVE_DATA_DIR, LOCAL_TEST)

print("データコピー完了")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 学習
SAVE_DIR = f"/content/runs/fold{FOLD}_size{SIZE:03d}"
!cd /content/anglist/train && python train.py \
    --data-dir {LOCAL_TRAIN} \
    --save-dir {SAVE_DIR} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --sigma {SIGMA} \
    --resize {RESIZE[0]} {RESIZE[1]}

In [ ]:
# ONNX エクスポート
!pip install -q onnx
!cd /content/anglist/train && python export_onnx.py \
    --checkpoint {SAVE_DIR}/best.pt \
    --output /content/model_raw.onnx \
    --height {RESIZE[0]} \
    --width {RESIZE[1]}

import onnx
proto = onnx.load("/content/model_raw.onnx", load_external_data=True)
onnx.save_model(proto, "/content/model.onnx", save_as_external_data=False)
print("ONNX export done")

In [ ]:
# テストセットでMRE評価
import subprocess, sys
sys.path.insert(0, "/content/anglist/train")

import json, math, os
import numpy as np
import onnxruntime as ort
import torch
from dataset import LANDMARK_ORDER, _percentile_clip_norm, _resize_with_padding

def preprocess(img_np, resize=(512,512)):
    if img_np.ndim == 3: img_np = img_np[0]
    img_np = _percentile_clip_norm(img_np)
    t = torch.from_numpy(img_np).unsqueeze(0)
    t, scale, pad_x, pad_y = _resize_with_padding(t, resize)
    return t.unsqueeze(0), scale, pad_x, pad_y

def postprocess(hm):
    hm = hm[0]
    coords = []
    for c in hm:
        idx = np.argmax(c)
        y, x = np.unravel_index(idx, c.shape)
        coords.append((float(x), float(y)))
    return coords

sess = ort.InferenceSession("/content/model.onnx", providers=["CPUExecutionProvider"])

all_errors = {name: [] for name in LANDMARK_ORDER}  # mm単位

for cid in test_ids:
    img_np = np.load(os.path.join(LOCAL_TEST, cid + "_image.npy"))
    inp_t, scale, pad_x, pad_y = preprocess(img_np)
    ort_out = sess.run(None, {"image": inp_t.numpy()})
    pred = postprocess(ort_out[0])

    with open(os.path.join(LOCAL_TEST, cid + "_landmarks.json")) as f:
        meta = json.load(f)
    spacing = meta["metadata"]["spacing"][0]
    lm = meta["landmarks_ijk"]

    for (px, py), name in zip(pred, LANDMARK_ORDER):
        gx, gy = lm[name]["i"], lm[name]["j"]
        x_o = (px - pad_x) / scale
        y_o = (py - pad_y) / scale
        re_mm = math.sqrt((x_o - gx)**2 + (y_o - gy)**2) * spacing
        all_errors[name].append(re_mm)

# 集計
results = {"fold": FOLD, "size": SIZE, "n_test": len(test_ids), "landmarks": {}}
all_vals = []
for name, errs in all_errors.items():
    mre = sum(errs) / len(errs)
    sdr2 = sum(1 for e in errs if e <= 2.0) / len(errs) * 100
    sdr4 = sum(1 for e in errs if e <= 4.0) / len(errs) * 100
    results["landmarks"][name] = {"mre_mm": mre, "sdr2": sdr2, "sdr4": sdr4}
    all_vals.extend(errs)
results["overall"] = {
    "mre_mm": sum(all_vals) / len(all_vals),
    "sdr2": sum(1 for e in all_vals if e <= 2.0) / len(all_vals) * 100,
    "sdr4": sum(1 for e in all_vals if e <= 4.0) / len(all_vals) * 100,
}

print(f"Overall MRE: {results['overall']['mre_mm']:.2f} mm | SDR@2mm: {results['overall']['sdr2']:.1f}% | SDR@4mm: {results['overall']['sdr4']:.1f}%")
for name, v in results["landmarks"].items():
    print(f"  {name}: MRE={v['mre_mm']:.2f}mm  SDR@2mm={v['sdr2']:.1f}%  SDR@4mm={v['sdr4']:.1f}%")

In [ ]:
# Driveに結果を保存
results_dir = os.path.join(DRIVE_LC_DIR, "results")
os.makedirs(results_dir, exist_ok=True)
out_path = os.path.join(results_dir, f"fold{FOLD}_size{SIZE:03d}.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"保存完了: {out_path}")
print("README.md の進捗チェックリストを更新してください。")